In [ ]:
### Good
import pulp as pl

def select_one_cell_per_basket_row_col_no_diag(
    baskets,
    n_rows,
    n_cols,
    exact_one_per_row_col=False,  # set True only if you want == 1 for each row/col
    solver=None
):
    """
    baskets: dict {b: [(i,j), ...]}
    Constraints:
      - exactly 1 cell per basket
      - at most 1 (or exactly 1) selected cell per row overall
      - at most 1 (or exactly 1) selected cell per column overall
      - no two selected cells are diagonally adjacent
    Returns:
      chosen: dict {b: (i,j)}
    """

    B = list(baskets.keys())

    # Decision vars: x[b,i,j] = 1 if basket b chooses cell (i,j)
    x = {}
    for b in B:
        for (i, j) in baskets[b]:
            x[(b, i, j)] = pl.LpVariable(f"x_{b}_{i}_{j}", 0, 1, cat=pl.LpBinary)

    prob = pl.LpProblem("OneCellPerBasket_UniqueRowCol_NoDiag", pl.LpMinimize)
    prob += 0  # feasibility objective

    # 1) Exactly one cell per basket
    for b in B:
        prob += pl.lpSum(x[(b, i, j)] for (i, j) in baskets[b]) == 1, f"one_cell_basket_{b}"

    # Create y[i,j] = 1 if any basket selects cell (i,j)
    y = {}
    for i in range(n_rows):
        for j in range(n_cols):
            y[(i, j)] = pl.LpVariable(f"y_{i}_{j}", 0, 1, cat=pl.LpBinary)
            # link: y(i,j) == sum_b x(b,i,j) over baskets that have this cell
            prob += y[(i, j)] == pl.lpSum(
                x[(b, i, j)] for b in B if (b, i, j) in x
            ), f"link_y_{i}_{j}"

    # 2) Row uniqueness
    for i in range(n_rows):
        row_sum = pl.lpSum(y[(i, j)] for j in range(n_cols))
        if exact_one_per_row_col:
            prob += row_sum == 1, f"row_exact_{i}"
        else:
            prob += row_sum <= 1, f"row_atmost_{i}"

    # 3) Column uniqueness
    for j in range(n_cols):
        col_sum = pl.lpSum(y[(i, j)] for i in range(n_rows))
        if exact_one_per_row_col:
            prob += col_sum == 1, f"col_exact_{j}"
        else:
            prob += col_sum <= 1, f"col_atmost_{j}"

    # 4) No diagonal adjacency:
    # forbid (i,j) with (i+1,j+1)
    for i in range(n_rows - 1):
        for j in range(n_cols - 1):
            prob += y[(i, j)] + y[(i + 1, j + 1)] <= 1, f"no_diag_downright_{i}_{j}"

    # forbid (i,j+1) with (i+1,j)
    for i in range(n_rows - 1):
        for j in range(n_cols - 1):
            prob += y[(i, j + 1)] + y[(i + 1, j)] <= 1, f"no_diag_downleft_{i}_{j}"

    # Solve
    if solver is None:
        solver = pl.PULP_CBC_CMD(msg=False)
    status = prob.solve(solver)

    if pl.LpStatus[status] != "Optimal":
        raise ValueError(f"No feasible solution found. Solver status: {pl.LpStatus[status]}")

    # Extract chosen cell per basket
    chosen = {}
    for b in B:
        for (i, j) in baskets[b]:
            if pl.value(x[(b, i, j)]) > 0.5:
                chosen[b] = (i, j)
                break

    return chosen

In [35]:
chosen = select_one_cell_per_basket_row_col_no_diag(
    baskets, 9, 9, exact_one_per_row_col=True
)
print(chosen)

{np.int32(5): (1, 0), np.int32(0): (0, 3), np.int32(2): (5, 4), np.int32(4): (2, 2), np.int32(3): (3, 5), np.int32(6): (4, 7), np.int32(7): (6, 6), np.int32(1): (7, 1), np.int32(8): (8, 8)}


In [36]:
chosen

{np.int32(5): (1, 0),
 np.int32(0): (0, 3),
 np.int32(2): (5, 4),
 np.int32(4): (2, 2),
 np.int32(3): (3, 5),
 np.int32(6): (4, 7),
 np.int32(7): (6, 6),
 np.int32(1): (7, 1),
 np.int32(8): (8, 8)}

In [14]:
import cv2
import numpy as np

IMG_PATH = "1.png"

def extract_grid_cells(image_path, grid_size=9):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    # Remove outer black border (important!)
    border = 5
    img = img[border:h-border, border:w-border]

    h, w = img.shape[:2]
    cell_h = h // grid_size
    cell_w = w // grid_size

    cells = {}
    colors = []

    for i in range(grid_size):
        for j in range(grid_size):
            y1 = i * cell_h
            y2 = (i + 1) * cell_h
            x1 = j * cell_w
            x2 = (j + 1) * cell_w

            cell = img[y1:y2, x1:x2]

            # Remove black grid lines
            mask = np.mean(cell, axis=2) > 40
            pixels = cell[mask]

            if len(pixels) == 0:
                mean_color = np.mean(cell.reshape(-1, 3), axis=0)
            else:
                mean_color = np.mean(pixels, axis=0)

            cells[(i, j)] = mean_color
            colors.append(mean_color)

    return cells, np.array(colors)


cells, colors = extract_grid_cells(IMG_PATH)

print("Total cells detected:", len(cells))  # Should be 81

Total cells detected: 81


In [23]:
from sklearn.cluster import KMeans

def create_baskets_from_colors(cells, colors, n_clusters=9):
    coords = list(cells.keys())

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(colors)

    baskets = {}
    for coord, label in zip(coords, labels):
        baskets.setdefault(label, []).append(coord)

    return baskets

baskets = create_baskets_from_colors(cells, colors)

print("Basket dictionary:")
print(baskets)

Basket dictionary:
{np.int32(5): [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (5, 1), (5, 2)], np.int32(0): [(0, 1), (0, 2), (0, 3), (0, 4), (1, 1), (1, 4), (2, 1), (3, 1), (3, 2), (3, 3), (4, 3), (5, 0), (5, 3), (6, 0), (6, 1), (6, 2), (6, 3)], np.int32(2): [(0, 5), (0, 6), (0, 7), (0, 8), (1, 3), (1, 5), (1, 6), (1, 7), (1, 8), (2, 3), (2, 4), (2, 5), (2, 8), (3, 4), (4, 4), (5, 4)], np.int32(4): [(1, 2), (2, 2)], np.int32(3): [(2, 6), (2, 7), (3, 5), (3, 6), (3, 7), (3, 8), (4, 5), (4, 8), (5, 5), (5, 8), (6, 5), (6, 8), (7, 5), (7, 6), (7, 7), (7, 8), (8, 6), (8, 7)], np.int32(6): [(4, 6), (4, 7), (5, 6)], np.int32(7): [(5, 7), (6, 6), (6, 7)], np.int32(1): [(6, 4), (7, 0), (7, 1), (7, 2), (7, 3), (7, 4), (8, 0), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5)], np.int32(8): [(8, 8)]}


c:\Users\roshn\anaconda3\envs\image_enhance\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [27]:
with open("baskets.txt", "w") as f:
    for key in sorted(baskets.keys()):
        f.write(f"Basket {key}:\n")
        for coord in baskets[key]:
            f.write(f"    {coord}\n")
        f.write("\n")

print("Dictionary saved to baskets.txt")

Dictionary saved to baskets.txt


### Code

In [1]:
import cv2
import numpy as np
from sklearn.cluster import KMeans
import pulp as pl

In [2]:
def extract_grid_cells(image_path, grid_size=9):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    # Remove outer black border (important!)
    border = 5
    img = img[border:h-border, border:w-border]

    h, w = img.shape[:2]
    cell_h = h // grid_size
    cell_w = w // grid_size

    cells = {}
    colors = []

    for i in range(grid_size):
        for j in range(grid_size):
            y1 = i * cell_h
            y2 = (i + 1) * cell_h
            x1 = j * cell_w
            x2 = (j + 1) * cell_w

            cell = img[y1:y2, x1:x2]

            # Remove black grid lines
            mask = np.mean(cell, axis=2) > 40
            pixels = cell[mask]

            if len(pixels) == 0:
                mean_color = np.mean(cell.reshape(-1, 3), axis=0)
            else:
                mean_color = np.mean(pixels, axis=0)

            cells[(i, j)] = mean_color
            colors.append(mean_color)

    return cells, np.array(colors)


In [3]:
def create_baskets_from_colors(cells, colors, n_clusters=9):
    coords = list(cells.keys())

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(colors)

    baskets = {}
    for coord, label in zip(coords, labels):
        baskets.setdefault(label, []).append(coord)

    return baskets

In [4]:
def select_one_cell_per_basket_row_col_no_diag(
    baskets,
    n_rows,
    n_cols,
    exact_one_per_row_col=False,  # set True only if you want == 1 for each row/col
    solver=None
):
    """
    baskets: dict {b: [(i,j), ...]}
    Constraints:
      - exactly 1 cell per basket
      - at most 1 (or exactly 1) selected cell per row overall
      - at most 1 (or exactly 1) selected cell per column overall
      - no two selected cells are diagonally adjacent
    Returns:
      chosen: dict {b: (i,j)}
    """

    B = list(baskets.keys())

    # Decision vars: x[b,i,j] = 1 if basket b chooses cell (i,j)
    x = {}
    for b in B:
        for (i, j) in baskets[b]:
            x[(b, i, j)] = pl.LpVariable(f"x_{b}_{i}_{j}", 0, 1, cat=pl.LpBinary)

    prob = pl.LpProblem("OneCellPerBasket_UniqueRowCol_NoDiag", pl.LpMinimize)
    prob += 0  # feasibility objective
    # 1) Exactly one cell per basket
    for b in B:
        prob += pl.lpSum(x[(b, i, j)] for (i, j) in baskets[b]) == 1, f"one_cell_basket_{b}"

    # Create y[i,j] = 1 if any basket selects cell (i,j)
    y = {}
    for i in range(n_rows):
        for j in range(n_cols):
            y[(i, j)] = pl.LpVariable(f"y_{i}_{j}", 0, 1, cat=pl.LpBinary)
            # link: y(i,j) == sum_b x(b,i,j) over baskets that have this cell
            prob += y[(i, j)] == pl.lpSum(
                x[(b, i, j)] for b in B if (b, i, j) in x
            ), f"link_y_{i}_{j}"

    # 2) Row uniqueness
    for i in range(n_rows):
        row_sum = pl.lpSum(y[(i, j)] for j in range(n_cols))
        if exact_one_per_row_col:
            prob += row_sum == 1, f"row_exact_{i}"
        else:
            prob += row_sum <= 1, f"row_atmost_{i}"
    # 3) Column uniqueness
    for j in range(n_cols):
        col_sum = pl.lpSum(y[(i, j)] for i in range(n_rows))
        if exact_one_per_row_col:
            prob += col_sum == 1, f"col_exact_{j}"
        else:
            prob += col_sum <= 1, f"col_atmost_{j}"

    # 4) No diagonal adjacency:
    # forbid (i,j) with (i+1,j+1)
    for i in range(n_rows - 1):
        for j in range(n_cols - 1):
            prob += y[(i, j)] + y[(i + 1, j + 1)] <= 1, f"no_diag_downright_{i}_{j}"

    # forbid (i,j+1) with (i+1,j)
    for i in range(n_rows - 1):
        for j in range(n_cols - 1):
            prob += y[(i, j + 1)] + y[(i + 1, j)] <= 1, f"no_diag_downleft_{i}_{j}"

    # Solve
    if solver is None:
        solver = pl.PULP_CBC_CMD(msg=False)
    status = prob.solve(solver)

    if pl.LpStatus[status] != "Optimal":
        raise ValueError(f"No feasible solution found. Solver status: {pl.LpStatus[status]}")

    # Extract chosen cell per basket
    chosen = {}
    for b in B:
        for (i, j) in baskets[b]:
            if pl.value(x[(b, i, j)]) > 0.5:
                chosen[b] = (i, j)
                break

    return chosen

In [ ]:
def draw_chosen_on_grid(
    image_path,
    chosen,                 # dict {basket_id: (i,j)}
    out_path="chosen_marked.png",
    grid_size=9,
    border=5,
    draw_cell_box=True
):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)

    H, W = img_bgr.shape[:2]

    # Crop outer border (same as extraction step)
    img_crop = img_bgr[border:H-border, border:W-border].copy()
    h, w = img_crop.shape[:2]
    cell_h = h // grid_size
    cell_w = w // grid_size

    # Optional: draw grid cell rectangles for clarity
    if draw_cell_box:
        for i in range(grid_size):
            for j in range(grid_size):
                y1, y2 = i * cell_h, (i + 1) * cell_h
                x1, x2 = j * cell_w, (j + 1) * cell_w
                cv2.rectangle(img_crop, (x1, y1), (x2, y2), (0, 0, 0), 1)

    # Mark chosen cells
    for basket_id, (i, j) in chosen.items():
        # cell bounds in the cropped image
        y1, y2 = i * cell_h, (i + 1) * cell_h
        x1, x2 = j * cell_w, (j + 1) * cell_w

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        # draw a filled white circle with black outline
        radius = int(min(cell_h, cell_w) * 0.18)
        cv2.circle(img_crop, (cx, cy), radius + 2, (0, 0, 0), thickness=-1)     # outline base
        cv2.circle(img_crop, (cx, cy), radius, (255, 255, 255), thickness=-1)   # inner fill

        # write basket id on top
        text = str(basket_id)
        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.6
        thickness = 2

        (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
        cv2.putText(
            img_crop,
            text,
            (cx - tw // 2, cy + th // 2),
            font,
            scale,
            (0, 0, 0),
            thickness,
            cv2.LINE_AA
        )

    # Paste crop back into original image so output matches original size
    out = img_bgr.copy()
    out[border:H-border, border:W-border] = img_crop

    cv2.imwrite(out_path, out)
    print(f"Saved marked image to: {out_path}")

In [ ]:
IMG_PATH = "1.png"
cells, colors = extract_grid_cells(IMG_PATH)
print("Total cells detected:", len(cells))  # Should be 81

baskets = create_baskets_from_colors(cells, colors)
with open("baskets.txt", "w") as f:
    for key in sorted(baskets.keys()):
        f.write(f"Basket {key}:\n")
        for coord in baskets[key]:
            f.write(f"    {coord}\n")
        f.write("\n")

print("Dictionary saved to baskets.txt")

chosen = select_one_cell_per_basket_row_col_no_diag(
    baskets, 9, 9, exact_one_per_row_col=True
)
print(chosen)
draw_chosen_on_grid(IMG_PATH, chosen, out_path="chosen_marked.png")

Total cells detected: 81


c:\Users\roshn\anaconda3\envs\image_enhance\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Dictionary saved to baskets.txt
{np.int32(5): (1, 0), np.int32(0): (0, 3), np.int32(2): (5, 4), np.int32(4): (2, 2), np.int32(3): (3, 5), np.int32(6): (4, 7), np.int32(7): (6, 6), np.int32(1): (7, 1), np.int32(8): (8, 8)}


In [8]:
chosen

{np.int32(5): (1, 0),
 np.int32(0): (0, 3),
 np.int32(2): (5, 4),
 np.int32(4): (2, 2),
 np.int32(3): (3, 5),
 np.int32(6): (4, 7),
 np.int32(7): (6, 6),
 np.int32(1): (7, 1),
 np.int32(8): (8, 8)}

In [12]:
import streamlit as st
import numpy as np
import cv2
from sklearn.cluster import KMeans
import pulp as pl
from pathlib import Path
import tempfile

ModuleNotFoundError: No module named 'streamlit'